# REINFORCE ve Baseline Teorisi

Bu notebook policy gradient ailesinin en temel algoritması olan REINFORCE'u detaylandırır. Actor-Critic, A2C, PPO gibi yöntemler bu temel fikrin varyans azaltılmış ve daha stabil hale getirilmiş devamlarıdır.

## 1. Amaç Fonksiyonu

REINFORCE on-policy bir algoritmadır. Gradyan tahmini, veriyi toplayan politikanın yine güncellenen politika olduğunu varsayar. Bu yüzden eski episode'ları uzun süre replay etmek teorik olarak doğru değildir; politika değiştikçe eski log olasılıklar yeni hedefi temsil etmez.

Parametreli stokastik politika:

$$\pi_\theta(a\mid s)$$

Amaç beklenen iskontolu toplam ödülü maksimize etmektir:

$$J(\theta)=\mathbb{E}_{\tau\sim\pi_\theta}[G_0]$$

$$G_t=\sum_{k=t}^{T-1}\gamma^{k-t}r_{k+1}$$

Burada trajectory olasılığı politika tarafından belirlenir. Ortam dinamiği bilinmek zorunda değildir; örneklenen episode'lar üzerinden gradyan tahmini yapılır.

## 2. Policy Gradient Teoremi'nin REINFORCE Hali

Log-derivative trick:

$$\nabla_\theta p_\theta(x)=p_\theta(x)\nabla_\theta\log p_\theta(x)$$

Bu kimlik kullanıldığında:

$$\nabla_\theta J(\theta)=\mathbb{E}\left[\sum_t \nabla_\theta\log\pi_\theta(a_t\mid s_t)G_t\right]$$

Minimize edilen pratik kayıp:

$$L(\theta)=-\frac{1}{B}\sum_t \log\pi_\theta(a_t\mid s_t)G_t$$

Pozitif return alınan aksiyonların log olasılığı artar; negatif veya ortalamanın altı return alan aksiyonların olasılığı azalır.

## 3. Reward-to-Go

Bir aksiyonun geçmişteki ödüllerden sorumlu tutulması gerekmez. Bu yüzden her zaman adımı için episode toplamı yerine sadece o andan sonraki return kullanılır:

$$G_t=r_{t+1}+\gamma r_{t+2}+\gamma^2r_{t+3}+...$$

Bu, bias eklemeden varyansı azaltır. REINFORCE'un standart pratik hali reward-to-go kullanır.

In [ ]:
import torch

def reward_to_go(rewards, dones, gamma=0.99):
    returns = torch.zeros_like(rewards)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running * (1.0 - dones[t])
        returns[t] = running
    return returns

rewards = torch.tensor([0.0, 0.0, 1.0, 0.0, -1.0])
dones = torch.tensor([0.0, 0.0, 1.0, 0.0, 1.0])
reward_to_go(rewards, dones)

## 4. Baseline Neden Bias Eklemiyor?

Duruma bağlı herhangi bir baseline çıkarılabilir:

$$\nabla_\theta J(\theta)=\mathbb{E}[\nabla_\theta\log\pi_\theta(a_t\mid s_t)(G_t-b(s_t))]$$

Çünkü:

$$\mathbb{E}_{a\sim\pi}[\nabla_\theta\log\pi_\theta(a\mid s)b(s)]=b(s)\nabla_\theta\sum_a\pi_\theta(a\mid s)=0$$

Yani baseline beklenen gradyanı değiştirmez, sadece örneklem varyansını azaltır.

## 5. En Yaygın Baseline: Value Function

$$b(s)=V_\phi(s)$$

Bu durumda öğrenme sinyali advantage olur:

$$\hat{A}_t=G_t-V_\phi(s_t)$$

Politika kaybı:

$$L_\pi(\theta)=-\log\pi_\theta(a_t\mid s_t)\hat{A}_t$$

Value kaybı:

$$L_V(\phi)=\frac{1}{2}(V_\phi(s_t)-G_t)^2$$

Bu yapı Actor-Critic'e geçişin ilk adımıdır.

## 6. REINFORCE + Baseline Algoritma Akışı

1. Mevcut politika $\pi_\theta$ ile bir veya daha fazla episode topla.
2. Her zaman adımı için reward-to-go $G_t$ hesapla.
3. Critic/baseline varsa $V_\phi(s_t)$ tahminini al.
4. Advantage hesapla: $\hat{A}_t=G_t-V_\phi(s_t)$.
5. Actor loss'u kur: $L_\pi=-\log\pi_\theta(a_t\mid s_t)\hat{A}_t$.
6. Value loss'u kur: $L_V=\frac{1}{2}(V_\phi(s_t)-G_t)^2$.
7. Toplam loss ile gradient descent adımı at.

Bu algoritma tam episode return kullandığı için basittir, ama uzun ve seyrek ödüllü ortamlarda yüksek varyanslıdır. Actor-Critic bu akışta $G_t$ yerine bootstrap edilmiş TD hedefleri kullanarak daha sık ve daha düşük varyanslı güncelleme yapar.

In [ ]:
def reinforce_with_baseline_loss(log_probs, returns, values):
    advantages = returns - values.detach()
    policy_loss = -(log_probs * advantages).mean()
    value_loss = 0.5 * (values - returns).pow(2).mean()
    return policy_loss, value_loss

log_probs = torch.tensor([-1.2, -0.7, -2.0])
returns = torch.tensor([1.0, 0.2, -0.5])
values = torch.tensor([0.4, 0.1, 0.0])
reinforce_with_baseline_loss(log_probs, returns, values)

## 6.1. REINFORCE İçin Policy + Baseline Ağı

Baseline ayrı bir tablo olmak zorunda değildir; çoğu uygulamada politika ağıyla ortak encoder kullanan küçük bir value head olarak kurulur. Bu yapı Actor-Critic'e benzer, fakat REINFORCE'ta hedef genellikle Monte Carlo return $G_t$'dir.

In [ ]:
try:
    import torch
    import torch.nn as nn
    from torch.distributions import Categorical
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Bu hücre için PyTorch gerekli. Jupyter'da orbit_wars_env kernelini seçin.") from exc

class ReinforceWithBaselineNet(nn.Module):
    """Ortak trunk + policy logits + baseline value."""
    def __init__(self, obs_dim=10, hidden_dim=64, n_actions=4):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
        )
        self.policy_head = nn.Linear(hidden_dim, n_actions)
        self.value_head = nn.Linear(hidden_dim, 1)

    def forward(self, obs):
        h = self.trunk(obs)
        logits = self.policy_head(h)
        value = self.value_head(h).squeeze(-1)
        return logits, value

net = ReinforceWithBaselineNet()
obs = torch.randn(6, 10)
returns = torch.tensor([1.2, 0.4, -0.3, 1.0, 0.0, 0.7])
logits, values = net(obs)
dist = Categorical(logits=logits)
actions = dist.sample()
log_probs = dist.log_prob(actions)

advantages = returns - values.detach()
policy_loss = -(log_probs * advantages).mean()
value_loss = 0.5 * (values - returns).pow(2).mean()
total_loss = policy_loss + value_loss

print("actions:", actions)
print("values:", values)
print("advantages:", advantages)
print("policy/value/total:", float(policy_loss.detach()), float(value_loss.detach()), float(total_loss.detach()))

## 6.2. Baseline’li REINFORCE Update Algoritması

Bu update'te actor ve baseline aynı optimizer ile güncellenir. Policy loss içinde advantage `detach` edilir; value head ise kendi MSE loss'u üzerinden return hedefine yaklaşır.

In [ ]:
def reinforce_baseline_update(net, optimizer, obs, actions, returns, value_coef=0.5):
    logits, values = net(obs)
    dist = Categorical(logits=logits)
    log_probs = dist.log_prob(actions)

    advantages = returns - values.detach()
    policy_loss = -(log_probs * advantages).mean()
    value_loss = 0.5 * (values - returns).pow(2).mean()
    loss = policy_loss + value_coef * value_loss

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    return {
        "loss": float(loss.detach()),
        "policy_loss": float(policy_loss.detach()),
        "value_loss": float(value_loss.detach()),
        "entropy": float(dist.entropy().mean().detach()),
    }

net = ReinforceWithBaselineNet(obs_dim=10, n_actions=4)
optimizer = torch.optim.Adam(net.parameters(), lr=1e-3)
obs = torch.randn(8, 10)
with torch.no_grad():
    logits, _ = net(obs)
    actions = Categorical(logits=logits).sample()
returns = torch.randn(8)
reinforce_baseline_update(net, optimizer, obs, actions, returns)

## 7. Avantaj Normalizasyonu

Pratikte advantage batch içinde normalize edilir:

$$\hat{A}\leftarrow\frac{\hat{A}-mean(\hat{A})}{std(\hat{A})+\epsilon}$$

Bu, gradyan ölçeğini daha stabil yapar. Optimumu teorik olarak değiştirmeyi hedeflemez; optimizasyonu daha iyi koşullandırır.

## 8. REINFORCE'un Sınırlamaları

| Sınırlama | Sonuç |
| --- | --- |
| Tam episode bekler | Uzun oyunlarda gecikmeli öğrenme |
| Monte Carlo return yüksek varyanslıdır | Çok noisy gradyan |
| On-policy'dir | Eski rollout'lar hızla geçersizleşir |
| Seyrek ödülde zayıftır | Başarı sinyali az gelir |

Bu sınırlamalar Actor-Critic, GAE ve PPO'nun neden ortaya çıktığını açıklar.